In [ ]:
## utils

import os
import random
import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def to_device(batch, device):
    if isinstance(batch, torch.Tensor):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(x, device) for x in batch)
    if isinstance(batch, dict):
        return {k: to_device(v, device) for k, v in batch.items()}
    return batch


def nan_safe_zscore(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    # z-score ignoring NaNs
    mu = np.nanmean(x)
    sig = np.nanstd(x)
    if not np.isfinite(mu):
        return np.zeros_like(x, dtype=np.float32)
    sig = max(sig, eps)
    return ((x - mu) / sig).astype(np.float32)


def nan_to_num(x: np.ndarray, fill: float = 0.0) -> np.ndarray:
    y = np.array(x, dtype=np.float32, copy=True)
    y[~np.isfinite(y)] = fill
    return y


In [ ]:
## metrics

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


def compute_metrics(y_true, y_pred, average="macro"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    return {
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "confusion_matrix": cm,
    }


In [ ]:
## windowing

import numpy as np


def segment_fixed_five_windows(x: np.ndarray, Tw: int = 33, overlap: float = 0.5, K: int = 5):
    """
    Implements the paper setup:
      - fixed K windows
      - window length Tw
      - 50% overlap -> stride = Tw*(1-overlap)

    If signal is too short, it pads with edge values (or zeros if empty).
    If signal is long, it takes the first K windows.
    """
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        x = np.zeros((Tw,), dtype=np.float32)

    stride = int(round(Tw * (1.0 - overlap)))
    stride = max(1, stride)

    needed_len = (K - 1) * stride + Tw
    if x.shape[0] < needed_len:
        pad_len = needed_len - x.shape[0]
        # edge-pad is often less destructive for physiological signals
        x = np.pad(x, (0, pad_len), mode="edge")

    windows = []
    for k in range(K):
        start = k * stride
        windows.append(x[start:start + Tw])
    return np.stack(windows, axis=0)  # [K, Tw]


In [ ]:
## augment

import numpy as np


def apply_lazy_eye(left: np.ndarray, right: np.ndarray, offset_mm: float, affect: str = "left"):
    out_l, out_r = left.copy(), right.copy()
    if affect == "left":
        out_l = out_l + offset_mm
    else:
        out_r = out_r + offset_mm
    return out_l, out_r


def apply_occlusion(x: np.ndarray, frac: float, mode: str = "nan"):
    """
    Replace a fraction of samples with NaN or saturation.
    """
    y = x.copy()
    n = y.shape[0]
    m = int(round(frac * n))
    if m <= 0:
        return y
    idx = np.random.choice(n, size=m, replace=False)
    if mode == "nan":
        y[idx] = np.nan
    else:
        # saturation value: use a high percentile as a plausible clipped level
        sat = np.nanpercentile(y, 95) if np.isfinite(np.nanmean(y)) else 0.0
        y[idx] = sat
    return y


def apply_illumination_bursts(x: np.ndarray, num_bursts: int, burst_len: int, sigma: float):
    """
    Inject Gaussian bursts (positive or negative).
    """
    y = x.copy()
    n = y.shape[0]
    if n == 0:
        return y
    for _ in range(num_bursts):
        start = np.random.randint(0, max(1, n - burst_len + 1))
        sign = 1.0 if np.random.rand() < 0.5 else -1.0
        noise = sign * np.random.normal(0.0, sigma, size=(burst_len,))
        y[start:start + burst_len] = y[start:start + burst_len] + noise.astype(np.float32)
    return y


def maybe_augment_pair(left, right, cfg):
    """
    Applies each disturbance with independent probability.
    """
    l, r = left.copy(), right.copy()

    # lazy eye
    if np.random.rand() < cfg["p_lazy_eye"]:
        off = np.random.uniform(cfg["lazy_eye_offset_mm_min"], cfg["lazy_eye_offset_mm_max"])
        affect = "left" if np.random.rand() < 0.5 else "right"
        l, r = apply_lazy_eye(l, r, offset_mm=float(off), affect=affect)

    # occlusion (one eye, random)
    if np.random.rand() < cfg["p_occlusion"]:
        frac = np.random.uniform(cfg["occlusion_frac_min"], cfg["occlusion_frac_max"])
        if np.random.rand() < 0.5:
            l = apply_occlusion(l, frac=float(frac), mode="nan")
        else:
            r = apply_occlusion(r, frac=float(frac), mode="nan")

    # illumination bursts (on mean / both eyes similarly)
    if np.random.rand() < cfg["p_illumination"]:
        sigma = np.random.uniform(cfg["illumination_burst_sigma_min"], cfg["illumination_burst_sigma_max"])
        nb = np.random.randint(cfg["illumination_num_bursts_min"], cfg["illumination_num_bursts_max"] + 1)
        bl = np.random.randint(cfg["illumination_burst_len_min"], cfg["illumination_burst_len_max"] + 1)
        l = apply_illumination_bursts(l, int(nb), int(bl), float(sigma))
        r = apply_illumination_bursts(r, int(nb), int(bl), float(sigma))

    return l, r


In [ ]:
## cwt

import numpy as np
import pywt


def compute_cwt_scalogram(x: np.ndarray, scales, wavelet: str):
    """
    Returns |CWT| scalogram as float32 array [S, T].
    NaNs are filled before transform.
    """
    x = np.asarray(x, dtype=np.float32)
    x_filled = x.copy()
    x_filled[~np.isfinite(x_filled)] = np.nanmean(x_filled) if np.isfinite(np.nanmean(x_filled)) else 0.0

    coeffs, _freqs = pywt.cwt(x_filled, scales=scales, wavelet=wavelet)
    # coeffs: [S, T] complex
    mag = np.abs(coeffs).astype(np.float32)
    return mag


In [ ]:
## data

import csv
import numpy as np
import torch
from torch.utils.data import Dataset

from .utils import nan_safe_zscore
from .augment import maybe_augment_pair
from .windowing import segment_fixed_five_windows


class NPZManifestDataset(Dataset):
    def __init__(self, manifest_csv, cfg_data, cfg_win, cfg_aug, train=True):
        self.rows = []
        with open(manifest_csv, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rows.append({"path": row["path"], "label": int(row["label"])})

        self.num_classes = int(cfg_data["num_classes"])
        self.use_skin_fallback = bool(cfg_data.get("use_skin_temp_fallback", False))

        self.Tw = int(cfg_win["Tw"])
        self.overlap = float(cfg_win["overlap"])
        self.K = int(cfg_win["num_windows"])

        self.train = bool(train)
        self.aug = cfg_aug
        self.aug_enable = bool(cfg_aug.get("enable", False)) and self.train

    def __len__(self):
        return len(self.rows)

    def _choose_input_signal(self, pl, pr, skin):
        """
        Decision strategy (simplified, robust):
        - if both pupils mostly valid: use mean
        - if one valid: use that
        - if both bad and fallback enabled: use skin
        """
        def valid_ratio(x):
            x = np.asarray(x)
            return float(np.mean(np.isfinite(x)))

        vr_l = valid_ratio(pl)
        vr_r = valid_ratio(pr)

        # "normal" threshold: you can tune this later
        thr = 0.7

        if vr_l >= thr and vr_r >= thr:
            x = np.nanmean(np.stack([pl, pr], axis=0), axis=0)
            src = "pupil_mean"
        elif vr_l >= thr:
            x = pl
            src = "pupil_left"
        elif vr_r >= thr:
            x = pr
            src = "pupil_right"
        else:
            if self.use_skin_fallback and skin is not None:
                x = skin
                src = "skin"
            else:
                # fall back to mean anyway
                x = np.nanmean(np.stack([pl, pr], axis=0), axis=0)
                src = "pupil_mean_fallback"
        return x, src

    def __getitem__(self, idx):
        row = self.rows[idx]
        label = row["label"]

        data = np.load(row["path"], allow_pickle=False)
        pl = data["pupil_left"].astype(np.float32)
        pr = data["pupil_right"].astype(np.float32)
        skin = data["skin_temp"].astype(np.float32) if ("skin_temp" in data.files) else None

        # augmentation (acts on pupils)
        if self.aug_enable:
            pl, pr = maybe_augment_pair(pl, pr, self.aug)

        x, src = self._choose_input_signal(pl, pr, skin)

        # per-subject z-score is ideal; here we do per-sample z-score (portable baseline)
        x = nan_safe_zscore(x)

        # fixed windowing
        xw = segment_fixed_five_windows(x, Tw=self.Tw, overlap=self.overlap, K=self.K)  # [K, Tw]

        return {
            "x_windows": torch.from_numpy(xw),  # float32
            "label": torch.tensor(label, dtype=torch.long),
            "source": src,
        }


In [ ]:
## model

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from .cwt import compute_cwt_scalogram


class FeedForward(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MHA(nn.Module):
    """
    Wrapper around torch MultiheadAttention for (B, N, D).
    """
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)

    def forward(self, x):
        y, _ = self.attn(x, x, x, need_weights=False)
        return y


class CrossAttn(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)

    def forward(self, q, kv):
        y, _ = self.attn(q, kv, kv, need_weights=False)
        return y


class TF_MFT_Block(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.ln_t = nn.LayerNorm(dim)
        self.ln_tf = nn.LayerNorm(dim)

        self.self_t = MHA(dim, heads, dropout)
        self.self_tf = MHA(dim, heads, dropout)

        self.cross_t_from_tf = CrossAttn(dim, heads, dropout)
        self.cross_tf_from_t = CrossAttn(dim, heads, dropout)

        self.gate_t = nn.Linear(dim, 1)
        self.gate_tf = nn.Linear(dim, 1)

        self.ln2_t = nn.LayerNorm(dim)
        self.ln2_tf = nn.LayerNorm(dim)
        self.ffn_t = FeedForward(dim, dropout)
        self.ffn_tf = FeedForward(dim, dropout)

    def forward(self, zt, ztf):
        # self-attn
        zt = zt + self.self_t(self.ln_t(zt))
        ztf = ztf + self.self_tf(self.ln_tf(ztf))

        # cross-attn
        ct = self.cross_t_from_tf(zt, ztf)
        ctf = self.cross_tf_from_t(ztf, zt)

        # adaptive gating (GAP over tokens)
        ut = zt.mean(dim=1)      # (B, D)
        utf = ztf.mean(dim=1)    # (B, D)

        gt = torch.sigmoid(self.gate_t(ut)).unsqueeze(1)    # (B, 1, 1)
        gtf = torch.sigmoid(self.gate_tf(utf)).unsqueeze(1)

        zt = zt + gt * ct
        ztf = ztf + gtf * ctf

        # FFN refinement
        zt = zt + self.ffn_t(self.ln2_t(zt))
        ztf = ztf + self.ffn_tf(self.ln2_tf(ztf))

        return zt, ztf


class TF_iPupil(nn.Module):
    def __init__(
        self,
        Tw: int,
        temporal_patch_len: int,
        cwt_scales,
        freq_patch_hw,
        dim: int = 96,
        heads: int = 4,
        low_layers: int = 4,
        high_layers: int = 2,
        num_classes: int = 3,
        dropout: float = 0.1,
        reliability_dim: int = 4,
        cwt_wavelet: str = "cmor1.5-1.0",
    ):
        super().__init__()
        self.Tw = Tw
        self.temporal_patch_len = temporal_patch_len
        self.scales = list(cwt_scales)
        self.wavelet = cwt_wavelet
        self.freq_patch_h, self.freq_patch_w = int(freq_patch_hw[0]), int(freq_patch_hw[1])
        self.dim = dim

        # temporal tokenization
        self.nt = math.ceil(Tw / temporal_patch_len)
        self.temporal_proj = nn.Linear(temporal_patch_len, dim)

        self.pos_t = nn.Parameter(torch.zeros(1, 1 + self.nt, dim))  # +1 for reliability token

        # tf tokenization
        self.S = len(self.scales)

        # We patchify scalogram [S, Tw] into blocks freq_patch_h x freq_patch_w (with padding)
        self.tf_proj = None  # set after computing Ntf from padded sizes

        # reliability embeddings
        self.rel_t = nn.Linear(2, dim)   # [Var(x), valid_ratio]
        self.rel_tf = nn.Linear(2, dim)  # [Var(CWT), mean(|CWT|)]

        # low-level TF-MFT blocks
        self.blocks = nn.ModuleList([TF_MFT_Block(dim, heads, dropout) for _ in range(low_layers)])

        # sequence-level transformer on window embeddings
        enc_layer = nn.TransformerEncoderLayer(
            d_model=2 * dim,
            nhead=heads,
            dim_feedforward=4 * (2 * dim),
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.seq_encoder = nn.TransformerEncoder(enc_layer, num_layers=high_layers)

        self.cls = nn.Sequential(
            nn.LayerNorm(2 * dim),
            nn.Linear(2 * dim, 2 * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * dim, num_classes),
        )

        self._init_tf_proj_and_pos()

    def _init_tf_proj_and_pos(self):
        # Determine padded TF sizes for patchify
        H = self.S
        W = self.Tw

        pad_h = (self.freq_patch_h - (H % self.freq_patch_h)) % self.freq_patch_h
        pad_w = (self.freq_patch_w - (W % self.freq_patch_w)) % self.freq_patch_w

        self.Hp = H + pad_h
        self.Wp = W + pad_w

        self.nh = self.Hp // self.freq_patch_h
        self.nw = self.Wp // self.freq_patch_w
        self.ntf = self.nh * self.nw

        patch_dim = self.freq_patch_h * self.freq_patch_w
        self.tf_proj = nn.Linear(patch_dim, self.dim)

        self.pos_tf = nn.Parameter(torch.zeros(1, 1 + self.ntf, self.dim))  # +1 for reliability token

        nn.init.trunc_normal_(self.pos_t, std=0.02)
        nn.init.trunc_normal_(self.pos_tf, std=0.02)

    def _tokenize_temporal(self, xw):
        """
        xw: (B, Tw)
        returns tokens: (B, 1+Nt, D) with reliability token at index 0
        """
        B, Tw = xw.shape
        pad = self.nt * self.temporal_patch_len - Tw
        if pad > 0:
            xw = F.pad(xw, (0, pad), mode="replicate")

        patches = xw.view(B, self.nt, self.temporal_patch_len)  # (B, Nt, Lt)
        xtok = self.temporal_proj(patches)                      # (B, Nt, D)

        # reliability features
        var = torch.var(xw, dim=1, unbiased=False)
        valid = torch.isfinite(xw).float().mean(dim=1)
        qt = torch.stack([var, valid], dim=1)                   # (B, 2)
        rt = self.rel_t(qt).unsqueeze(1)                        # (B, 1, D)

        zt = torch.cat([rt, xtok], dim=1)
        zt = zt + self.pos_t[:, : zt.size(1)]
        return zt

    def _patchify_tf(self, scalogram):
        """
        scalogram: (B, S, Tw) magnitude
        returns tokens: (B, 1+Ntf, D)
        """
        B, S, Tw = scalogram.shape

        # pad to (Hp, Wp)
        pad_h = self.Hp - S
        pad_w = self.Wp - Tw
        if pad_h > 0 or pad_w > 0:
            scalogram = F.pad(scalogram, (0, pad_w, 0, pad_h), mode="replicate")

        # patchify
        x = scalogram.view(B, self.nh, self.freq_patch_h, self.nw, self.freq_patch_w)
        x = x.permute(0, 1, 3, 2, 4).contiguous()  # (B, nh, nw, ph, pw)
        x = x.view(B, self.ntf, self.freq_patch_h * self.freq_patch_w)  # (B, Ntf, patch_dim)
        xtok = self.tf_proj(x)  # (B, Ntf, D)

        # reliability features on TF
        var = torch.var(scalogram, dim=(1, 2), unbiased=False)
        mean_mag = scalogram.mean(dim=(1, 2))
        qtf = torch.stack([var, mean_mag], dim=1)
        rtf = self.rel_tf(qtf).unsqueeze(1)

        ztf = torch.cat([rtf, xtok], dim=1)
        ztf = ztf + self.pos_tf[:, : ztf.size(1)]
        return ztf

    @torch.no_grad()
    def _batch_cwt(self, xw_cpu):
        """
        xw_cpu: numpy (B, Tw)
        returns torch (B, S, Tw)
        """
        mags = []
        for i in range(xw_cpu.shape[0]):
            mag = compute_cwt_scalogram(xw_cpu[i], scales=self.scales, wavelet=self.wavelet)
            mags.append(mag)
        mags = torch.from_numpy(torch.tensor(mags).numpy())  # safe conversion
        return mags

    def forward(self, x_windows):
        """
        x_windows: (B, K, Tw)
        """
        B, K, Tw = x_windows.shape
        device = x_windows.device

        window_embs = []
        for k in range(K):
            xw = x_windows[:, k, :]  # (B, Tw)

            # temporal tokens
            zt = self._tokenize_temporal(xw)

            # TF tokens via CWT (CPU for pywt, then back)
            xw_np = xw.detach().cpu().numpy()
            mag = []
            for i in range(B):
                mag_i = compute_cwt_scalogram(xw_np[i], scales=self.scales, wavelet=self.wavelet)
                mag.append(mag_i)
            mag = torch.from_numpy(torch.stack([torch.from_numpy(m) for m in mag]).numpy()).to(device)
            # mag: (B, S, Tw)

            ztf = self._patchify_tf(mag)

            # low-level fusion blocks
            for blk in self.blocks:
                zt, ztf = blk(zt, ztf)

            # window embedding (Eq. 22)
            et = zt.mean(dim=1)
            etf = ztf.mean(dim=1)
            e = torch.cat([et, etf], dim=1)  # (B, 2D)
            window_embs.append(e)

        U = torch.stack(window_embs, dim=1)  # (B, K, 2D)

        U = self.seq_encoder(U)              # (B, K, 2D)
        pooled = U.mean(dim=1)               # (B, 2D)
        logits = self.cls(pooled)            # (B, C)
        return logits


In [ ]:
## train

import os
import yaml
import time
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from .utils import set_seed, ensure_dir, to_device
from .dataset import NPZManifestDataset
from .model import TF_iPupil
from .metrics import compute_metrics


def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


def train_one_epoch(model, loader, opt, scaler, device, grad_clip=1.0, log_every=50):
    model.train()
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    y_true, y_pred = [], []

    pbar = tqdm(loader, desc="train", leave=False)
    for step, batch in enumerate(pbar, 1):
        batch = to_device(batch, device)
        x = batch["x_windows"]
        y = batch["label"]

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            logits = model(x)
            loss = loss_fn(logits, y)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

        losses.append(loss.item())
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(pred)

        if step % log_every == 0:
            pbar.set_postfix(loss=float(np.mean(losses[-log_every:])))

    metrics = compute_metrics(y_true, y_pred, average="macro")
    return float(np.mean(losses)), metrics


@torch.no_grad()
def eval_one_epoch(model, loader, device):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    y_true, y_pred = [], []

    for batch in tqdm(loader, desc="eval", leave=False):
        batch = to_device(batch, device)
        x = batch["x_windows"]
        y = batch["label"]

        logits = model(x)
        loss = loss_fn(logits, y)

        losses.append(loss.item())
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(pred)

    metrics = compute_metrics(y_true, y_pred, average="macro")
    return float(np.mean(losses)), metrics


def main(cfg_path="configs/default.yaml"):
    cfg = load_yaml(cfg_path)

    set_seed(int(cfg["seed"]))
    device = torch.device(cfg.get("device", "cuda") if torch.cuda.is_available() else "cpu")

    train_ds = NPZManifestDataset(
        cfg["data"]["train_manifest"],
        cfg["data"],
        cfg["windowing"],
        cfg["augment"],
        train=True,
    )
    test_ds = NPZManifestDataset(
        cfg["data"]["test_manifest"],
        cfg["data"],
        cfg["windowing"],
        cfg["augment"],
        train=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["train"]["batch_size"]),
        shuffle=True,
        num_workers=int(cfg["train"]["num_workers"]),
        pin_memory=True,
        drop_last=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=int(cfg["train"]["batch_size"]),
        shuffle=False,
        num_workers=int(cfg["train"]["num_workers"]),
        pin_memory=True,
        drop_last=False,
    )

    mcfg = cfg["model"]
    model = TF_iPupil(
        Tw=int(cfg["windowing"]["Tw"]),
        temporal_patch_len=int(mcfg["temporal_patch_len"]),
        cwt_scales=cfg["cwt"]["scales"],
        freq_patch_hw=mcfg["freq_patch_hw"],
        dim=int(mcfg["dim"]),
        heads=int(mcfg["heads"]),
        low_layers=int(mcfg["low_layers"]),
        high_layers=int(mcfg["high_layers"]),
        num_classes=int(cfg["data"]["num_classes"]),
        dropout=float(mcfg["dropout"]),
        reliability_dim=int(mcfg["reliability_dim"]),
        cwt_wavelet=str(cfg["cwt"]["wavelet"]),
    ).to(device)

    opt = AdamW(
        model.parameters(),
        lr=float(cfg["train"]["lr"]),
        weight_decay=float(cfg["train"]["weight_decay"]),
    )
    sched = CosineAnnealingLR(opt, T_max=int(cfg["train"]["epochs"]))

    use_amp = (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    save_dir = cfg["train"]["save_dir"]
    ensure_dir(save_dir)

    best_acc = -1.0
    for epoch in range(1, int(cfg["train"]["epochs"]) + 1):
        t0 = time.time()

        tr_loss, tr_m = train_one_epoch(
            model, train_loader, opt, scaler if use_amp else None, device,
            grad_clip=float(cfg["train"]["grad_clip"]),
            log_every=int(cfg["train"]["log_every"]),
        )
        te_loss, te_m = eval_one_epoch(model, test_loader, device)
        sched.step()

        acc = te_m["accuracy"]
        dt = time.time() - t0

        print(
            f"epoch {epoch:03d} | "
            f"train loss {tr_loss:.4f} acc {tr_m['accuracy']:.4f} | "
            f"test loss {te_loss:.4f} acc {acc:.4f} f1 {te_m['f1']:.4f} | "
            f"time {dt:.1f}s"
        )

        if acc > best_acc:
            best_acc = acc
            ckpt = {
                "epoch": epoch,
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "cfg": cfg,
                "best_acc": best_acc,
            }
            torch.save(ckpt, os.path.join(save_dir, "best.pt"))
            print(f"saved best checkpoint (acc={best_acc:.4f})")

    # final save
    torch.save({"model": model.state_dict(), "cfg": cfg}, os.path.join(save_dir, "last.pt"))
    print("done.")


if __name__ == "__main__":
    main()


In [ ]:
## eval

import yaml
import numpy as np
import torch
from torch.utils.data import DataLoader

from .dataset import NPZManifestDataset
from .model import TF_iPupil
from .metrics import compute_metrics
from .utils import to_device


def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


@torch.no_grad()
def main(cfg_path="configs/default.yaml", ckpt_path="runs/tf_ipupil/best.pt"):
    cfg = load_yaml(cfg_path)
    device = torch.device(cfg.get("device", "cuda") if torch.cuda.is_available() else "cpu")

    ds = NPZManifestDataset(
        cfg["data"]["test_manifest"],
        cfg["data"],
        cfg["windowing"],
        cfg["augment"],
        train=False,
    )
    loader = DataLoader(ds, batch_size=int(cfg["train"]["batch_size"]), shuffle=False, num_workers=2)

    mcfg = cfg["model"]
    model = TF_iPupil(
        Tw=int(cfg["windowing"]["Tw"]),
        temporal_patch_len=int(mcfg["temporal_patch_len"]),
        cwt_scales=cfg["cwt"]["scales"],
        freq_patch_hw=mcfg["freq_patch_hw"],
        dim=int(mcfg["dim"]),
        heads=int(mcfg["heads"]),
        low_layers=int(mcfg["low_layers"]),
        high_layers=int(mcfg["high_layers"]),
        num_classes=int(cfg["data"]["num_classes"]),
        dropout=float(mcfg["dropout"]),
        reliability_dim=int(mcfg["reliability_dim"]),
        cwt_wavelet=str(cfg["cwt"]["wavelet"]),
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    y_true, y_pred = [], []
    for batch in loader:
        batch = to_device(batch, device)
        logits = model(batch["x_windows"])
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
        y_true.extend(batch["label"].detach().cpu().numpy().tolist())
        y_pred.extend(pred)

    m = compute_metrics(y_true, y_pred)
    print("accuracy:", m["accuracy"])
    print("precision:", m["precision"])
    print("recall:", m["recall"])
    print("f1:", m["f1"])
    print("confusion_matrix:\n", m["confusion_matrix"])


if __name__ == "__main__":
    main()
